In [1]:
import pathlib as pl
import json as json
import socket as skt
import functools as fnt
import datetime as dt

@fnt.cache
def load_machine_project_config(machine):

    wd = pl.Path(".").resolve()
    while 1:
        try:
            cfg_file = wd.joinpath("project-config.json").resolve(strict=True)
            break
        except FileNotFoundError:
            wd = wd.parent
            if wd.name == "project-chrom-y-extended":
                raise RuntimeError("Leaving project context")

    _PROJECT_CONFIG = json.load(open(cfg_file))
    machine_config = _PROJECT_CONFIG[machine]
    _config = _PROJECT_CONFIG["GENERIC"]
    _config.update(machine_config)

    config = dict()
    for key, value in _config.items():
        if isinstance(value, str) and "/" in value:
            config[key] = pl.Path(value)
        else:
            config[key] = value

    return config


MACHINE = skt.gethostname()
CONFIG = load_machine_project_config(MACHINE)

TIMESTAMP = dt.datetime.today().strftime("%Y%m%dT%H%M")
TS = TIMESTAMP


def replace_path_prefix(path, local_to_remote=True, infrastructure="hilbert", path_config=CONFIG):

    local_prefix = str(CONFIG[f"local_{infrastructure}_prefix"])
    remote_prefix = str(CONFIG[f"remote_{infrastructure}_prefix"])
    if local_to_remote:
        path = str(path).replace(local_prefix, remote_prefix)
    else:
        path = str(path).replace(remote_prefix, local_prefix)

    return path